# Notebook 1 — Text QA Pipeline

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

## What you'll build

By the end of this notebook you'll have a small **reading-comprehension QA dataset** grounded in real Wikipedia excerpts, with each row scored by an LLM judge and quality-filtered.

A finished row will look roughly like this:

| title | difficulty | question | answer | judge |
|---|---|---|---|---|
| Albedo | medium | How does albedo describe the amount of sunlight a body reflects? | `{ "text": "Albedo is the fraction of sunlight diffusely reflected by a body, measured from 0 to 1.", "justification": "The excerpt defines albedo as the fraction of sunlight reflected by a body." }` | `{ "faithfulness": { "score": 5 }, "completeness": { "score": 5 } }` |

## What you'll learn

1. **Controlled sampling** — sampler columns produce structured metadata *without* an LLM. Fast, deterministic, and the right way to steer downstream LLM diversity.
2. **Seed datasets** — bring real text into the pipeline so generations are grounded, not invented.
3. **LLM columns with Jinja templating** — `{{ column_name }}` in a prompt resolves to that row's value. The framework figures out the dependency DAG.
4. **Structured outputs** — Pydantic schemas give you typed, validated fields per row.
5. **LLM-as-a-judge** — score each row on rubrics, then filter. Your training set quality is a function of how aggressively you filter.
6. **Preview → Create loop** — iterate on small samples, scale up only when you're happy.

## Pipeline shape

```
Wikipedia excerpts (seed)
        │
        ▼
   ┌────────────────────────────────┐
   │ topic / difficulty / qtype     │  sampler columns (no LLM)
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ question                       │  LLM text column
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ answer                         │  LLM structured column (Pydantic)
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ judge (faithfulness, completeness) │  LLM judge column
   └────────────────────────────────┘
        │
        ▼
    filter to score ≥ 4 → quality-filtered QA dataset
```

## Setup

Imports, environment variables, and a quick check of the model provider you've configured. Most logic lives in `notebook_helpers.py` so the notebook reads as narrative.

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner
from notebook_helpers import (
    DATA_DIR,
    build_dd_model_setup,
    environment_setup,
    load_wiki_excerpts,
    show_provider_info,
)
from prompts import (
    ANSWER_PROMPT,
    JUDGE_COMPLETENESS_DESCRIPTION,
    JUDGE_COMPLETENESS_OPTIONS,
    JUDGE_FAITHFULNESS_DESCRIPTION,
    JUDGE_FAITHFULNESS_OPTIONS,
    JUDGE_PROMPT,
    QUESTION_PROMPT,
)
from pydantic import BaseModel, Field

# Pick which hosted backend to use. Leave as "auto" to use whichever key is in
# your .env (precedence: NVIDIA -> OpenRouter -> OpenAI), or set explicitly to
# "nvidia", "openrouter", or "openai" to flip between providers without editing
# any other files.
PROVIDER = "auto"

provider = environment_setup(provider=PROVIDER)
show_provider_info(provider)


model_providers, model_configs = build_dd_model_setup(provider)
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
data_designer = DataDesigner(model_providers=model_providers)


## Part 1 — Controlled sampling (no LLM yet)

Two key objects drive everything in Data Designer:

- **`DataDesignerConfigBuilder`** — fluent API for defining your dataset schema, one column at a time.
- **`DataDesigner`** — the runtime that validates the config, previews small samples, and creates full datasets.

We'll start by adding **sampler columns**: structured metadata produced without any LLM call. Sampler columns are how you control the *distribution* of your training set.

### Sampler columns

We define three:

- **`question_difficulty`** — categorical, weighted: 30% easy, 50% medium, 20% hard. Skews the dataset toward the medium range where most useful training signal lives.
- **`question_type`** — *subcategory* sampler: depends on `question_difficulty`. Easy questions get factual / definition types; hard questions get inference / comparison. This is how you encode *correlations* between fields without writing if/else.
- **`document_type`** — categorical with weights. Adds another axis of diversity for stratified analysis later.

No LLM has been called yet.

In [ ]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="question_difficulty",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["easy", "medium", "hard"],
            weights=[0.3, 0.5, 0.2],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="question_type",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="question_difficulty",
            values={
                "easy":   ["factual", "definition"],
                "medium": ["factual", "inference"],
                "hard":   ["inference", "comparison"],
            },
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="document_type",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["article", "report", "manual", "memo"],
            weights=[0.4, 0.3, 0.2, 0.1],
        ),
    )
)

data_designer.validate(config_builder)
print("Sampler config validated.")

### Preview the sampler-only dataset

`preview()` runs the pipeline on a small number of rows. Since we only have sampler columns so far, this is instant and free.

In [ ]:
preview = data_designer.preview(config_builder, num_records=20)
preview.dataset.head(10)

In [ ]:
preview.dataset[["question_difficulty", "question_type", "document_type"]].describe()

🔍 **Check the sampler distributions.** The goal here is to confirm that the generated rows roughly follow the weights you configured. With only 20 preview rows the percentages will be noisy; at larger scale they should move closer to the target weights. Also notice that `question_type` depends on `question_difficulty` — that's the subcategory sampler encoding a correlation between fields.

## Part 2 — Add a seed dataset

Sampling alone gives you metadata diversity, but the questions need to be *about* something. We'll layer in a **seed dataset**: a small checked-in parquet sample of Wikipedia excerpts derived from the [Wikimedia Wikipedia dataset on Hugging Face](https://huggingface.co/datasets/wikimedia/wikipedia). Every generated row will be grounded in one of these excerpts, so questions are answerable from real text.

In [ ]:
wiki = load_wiki_excerpts()
print(f"Loaded {len(wiki)} wiki excerpts.\n")
wiki.head(3)

In [ ]:
config_builder.with_seed_dataset(
    seed_source=dd.LocalFileSeedSource(
        path=str(DATA_DIR / "wiki_seed.parquet"),
    ),
    sampling_strategy=dd.SamplingStrategy.SHUFFLE,
)

Each seed row will be available as columns by their JSON key — `article`, `title`, `topic`, `url`. Our prompts (defined in `prompts.py`) reference `{{ article }}` directly.

## Model providers and configs

So far we have only used sampler columns and a seed dataset. The next cells start calling an LLM, so this is the moment to inspect how models are wired into Data Designer.

- **Model providers** tell Data Designer where inference requests should go, including the API endpoint and which environment variable contains the key.
- **Model configs** tell Data Designer which concrete models to use, plus friendly aliases like `text-llm` and `vlm`.

Columns refer to aliases, not provider-specific model IDs. That lets the same notebook run against NVIDIA Build, OpenRouter, or OpenAI by changing the setup cell, while the dataset pipeline stays the same.


In [ ]:
# Data Designer ships with default provider and model config definitions.
DataDesigner().info.display(dd.InfoType.MODEL_PROVIDERS)
dd.DataDesignerConfigBuilder().info.display(dd.InfoType.MODEL_CONFIGS)


In [ ]:
data_designer.info.display(dd.InfoType.MODEL_PROVIDERS)
config_builder.info.display(dd.InfoType.MODEL_CONFIGS)


### Build your own provider and config

The helper above keeps the workshop portable, but the underlying objects are plain Data Designer configs. In your own project, you can create a provider and one or more model configs directly. The provider names the endpoint and secret; each model config gives a model a stable alias that columns can reference.


In [ ]:
custom_provider = dd.ModelProvider(
    name="my-openai-compatible-provider",
    endpoint="https://api.example.com/v1",
    provider_type="openai",
    api_key="MY_PROVIDER_API_KEY",
    extra_body={"reasoning_effort": "medium"},
)

custom_text_model = dd.ModelConfig(
    alias="text-llm",
    model="provider/model-name",
    provider="my-openai-compatible-provider",
    inference_parameters=dd.ChatCompletionInferenceParams(
        timeout=120,
        max_tokens=4096,
        max_parallel_requests=8,
    ),
)

custom_judge_model = dd.ModelConfig(
    alias="judge-llm",
    model="provider/model-name",
    provider="my-openai-compatible-provider",
    inference_parameters=dd.ChatCompletionInferenceParams(
        timeout=120,
        max_tokens=4096,
        max_parallel_requests=8,
        temperature=0.0,
    ),
)

# In a real notebook, you would wire these in like this:
# custom_data_designer = DataDesigner(model_providers=[custom_provider])
# custom_config_builder = dd.DataDesignerConfigBuilder(
#     model_configs=[custom_text_model, custom_judge_model]
# )

print("Custom provider/configs are not used below; they show the shape you would pass in.")


## Part 3 — LLM-generated text + judge

Now we add the columns that actually call the model. `LLMTextColumnConfig` sends a prompt and stores the response. Prompts use **Jinja templating** so each row gets its own contextualised generation:

- `{{ article }}` — the seed excerpt for this row
- `{{ question_difficulty }}`, `{{ question_type }}` — the sampler values

Data Designer figures out the dependency graph automatically: `question` depends on `article`, `question_difficulty`, and `question_type`; the framework generates `question` only after all three are available.

In [ ]:
# Inspect the prompt before adding the column.
print(QUESTION_PROMPT)

In [ ]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="question",
        prompt=QUESTION_PROMPT,
        model_alias=provider.text_alias,
    )
)

### Structured output for the answer

For the answer we want more than free text — we want a typed object. `LLMStructuredColumnConfig` constrains output to a Pydantic schema. Each row will have an `answer.text` and `answer.justification` we can score later.

We also turn on `with_trace=ALL_MESSAGES` so we can replay the exact messages sent to the model for debugging.

In [ ]:
class Answer(BaseModel):
    text: str = Field(description="The answer to the question, grounded in the article.")
    justification: str = Field(
        description="One sentence quoting or paraphrasing the supporting evidence."
    )


config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="answer",
        prompt=ANSWER_PROMPT,
        model_alias=provider.text_alias,
        output_format=Answer,
        with_trace=dd.TraceType.ALL_MESSAGES,
    )
)

data_designer.validate(config_builder)
print("Question and answer config validated.")

### LLM as judge

`LLMJudgeColumnConfig` scores each row against the rubrics you define. Our judge scores two things:

- **faithfulness** — does the answer trace back to the excerpt?
- **completeness** — does the answer fully address the question?

Each on a 1–5 scale. The judge runs as part of the same pipeline, so the preview will already include scores.

We use a dedicated `judge-llm` alias that points at the same text model as `text-llm`, but with `temperature=0.0` for more stable scoring.

In [ ]:
config_builder.add_column(
    dd.LLMJudgeColumnConfig(
        name="judge",
        prompt=JUDGE_PROMPT,
        model_alias=provider.judge_alias,
        scores=[
            dd.Score(
                name="faithfulness",
                description=JUDGE_FAITHFULNESS_DESCRIPTION,
                options=JUDGE_FAITHFULNESS_OPTIONS,
            ),
            dd.Score(
                name="completeness",
                description=JUDGE_COMPLETENESS_DESCRIPTION,
                options=JUDGE_COMPLETENESS_OPTIONS,
            ),
        ],
    )
)

data_designer.validate(config_builder)
print("Full QA pipeline config validated.")

### Preview (includes judge scores)

The preview runs every column — question, answer, **and** the judge — so you can see the full pipeline in one shot. This is the inner loop: small `preview` calls, inspect outputs, iterate on prompts and rubrics.

**`preview.display_sample_record()`** is the right tool here, not a flat dataframe view:

- Each call shows **one** record at a time, fully expanded — no truncation of structured fields.
- Calling it again **rotates to the next record** (just hit `Shift+Enter` repeatedly on the same cell), so you can inspect a handful of rows without writing a loop.
- It includes the **LLM conversation traces** for every column that was generated with `with_trace=...`. The exact messages sent to the model are right there with the output — no separate inspection step needed.
- Pass `index=N` if you want to pin to a specific row.

In [ ]:
preview = data_designer.preview(config_builder, num_records=3)

In [ ]:
# Re-run this cell (Shift+Enter) to cycle through records 1, 2, 3, 1, 2, 3...
preview.display_sample_record()

In [ ]:
# Pass `index=N` to pin to a specific row — handy for referring back to
# "row 0" while iterating on a prompt or debugging a specific failure.
preview.display_sample_record(index=0)

## Part 4 — Scale up and filter

`create()` runs the pipeline at full size. Set `num_records` modestly during the workshop to keep API costs down — bump it later when you're using the pipeline for real work.

In [ ]:
results = data_designer.create(
    config_builder,
    num_records=20,
    dataset_name="01_text_qa",
)
df = results.load_dataset()
print(f"Generated {len(df)} rows.")
df.head()

### Filter on judge scores

Keep only rows where **both** faithfulness and completeness ≥ 4. This is your training-quality filter; tune the threshold to your tolerance.

In [ ]:
def keep(row):
    judge = row["judge"]
    if not isinstance(judge, dict):
        return False
    faith = int(judge.get("faithfulness", {}).get("score", 0))
    comp  = int(judge.get("completeness", {}).get("score", 0))
    return faith >= 4 and comp >= 4

filtered = df[df.apply(keep, axis=1)].reset_index(drop=True)
print(f"Kept {len(filtered)} / {len(df)} rows ({len(filtered) / len(df):.0%} keep rate).")
filtered[["title", "question_difficulty", "question", "answer"]].head()

### Optional: publish the create result to Hugging Face Hub

The `results` object returned by `create()` can publish the full Data Designer run directly: generated parquet batches, builder config, metadata, and an autogenerated dataset card. That is usually better than rebuilding a separate Hugging Face `Dataset`, because the Hub copy preserves the pipeline context.

The cell is disabled by default so nobody accidentally publishes workshop data from a shared environment. To run it, log in with `huggingface-cli login` or set `HF_TOKEN`, choose your repo ID, and flip `PUSH_TO_HUB` to `True`.


In [ ]:
PUSH_TO_HUB = False
HF_REPO_ID = "your-username/pydata-text-qa-demo"
HUB_DESCRIPTION = """Synthetic reading-comprehension QA examples generated with
Data Designer for the PyData London 2026 workshop."""

if PUSH_TO_HUB:
    hub_url = results.push_to_hub(
        repo_id=HF_REPO_ID,
        description=HUB_DESCRIPTION,
        private=True,
        tags=["data-designer", "synthetic-data", "pydata-london-2026"],
    )
    print(f"Pushed Data Designer run artifacts to {hub_url}")
else:
    print("Set PUSH_TO_HUB = True and HF_REPO_ID to publish this create result.")


## Recap

You built a four-stage pipeline:

**sample → seed → generate + judge → filter**

All declarative. No prompt-script soup. Easy to bump from 20 rows to 20K — the only thing that changes is `num_records`.

## Next

👉 [`02_document_visual_qa.ipynb`](02_document_visual_qa.ipynb) — the same pipeline shape, but the seed data is **rich synthetic document images** and the model is a **VLM**. The judge becomes a multimodal judge that has to inspect the page to score the answer.